# Guardar modelo para producción

**Objetivo:** entrenar el modelo final y guardarlo como archivo `.pkl` para que la API lo cargue sin re-entrenar.

**¿Por qué guardamos el modelo?**
En producción nunca re-entrenamos el modelo cuando la API arranca — eso tomaría minutos
y haría la API inutilizable. En cambio, entrenamos una vez, guardamos el resultado,
y la API simplemente carga ese archivo en memoria al iniciar.

**¿Qué guardamos?**
- El modelo LightGBM entrenado
- La lista de features exactas (orden y nombres) que el modelo espera
- Los metadatos (ROC-AUC, número de árboles) para trazabilidad

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

RANDOM_STATE = 42
DATA_PATH = '../data/raw/application_train.csv'
MODELS_DIR = Path('../models')

print(f'Guardando modelos en: {MODELS_DIR.resolve()}')

Guardando modelos en: /Users/angelasap107/Trabajo/Portafolio/credit-risk-homecredit/models


## 1. Preprocesamiento y entrenamiento final

In [2]:
df = pd.read_csv(DATA_PATH)

missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.4].index.tolist()
df = df.drop(columns=cols_to_drop)

df['AGE_YEARS'] = df['DAYS_BIRTH'].abs() / 365
df = df.drop(columns=['DAYS_BIRTH'])

df['FLAG_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

df = df.drop(columns=['SK_ID_CURR'])

cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Guardamos la lista de features en el orden exacto que el modelo espera
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

print('Entrenando modelo final...')
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

y_proba = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba)

print(f'\nÁrboles usados: {model.best_iteration_}')
print(f'ROC-AUC final:  {roc_auc:.4f}')

Entrenando modelo final...
[100]	valid_0's auc: 0.753176
[200]	valid_0's auc: 0.755528
[300]	valid_0's auc: 0.755867

Árboles usados: 345
ROC-AUC final:  0.7561


## 2. Guardar modelo y metadatos

In [3]:
# Guardar el modelo
model_path = MODELS_DIR / 'lgbm_credit_risk.pkl'
joblib.dump(model, model_path)
print(f'Modelo guardado: {model_path}')

# Guardar metadatos en JSON para trazabilidad
metadata = {
    'model': 'LGBMClassifier',
    'roc_auc': round(roc_auc, 4),
    'best_iteration': model.best_iteration_,
    'n_features': len(feature_names),
    'feature_names': feature_names,
    'params': {
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'scale_pos_weight': round(scale_pos_weight, 4),
        'metric': 'auc'
    },
    'preprocessing': {
        'drop_missing_threshold': 0.4,
        'cols_dropped': len(cols_to_drop),
        'categorical_cols': cat_cols
    }
}

metadata_path = MODELS_DIR / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadatos guardados: {metadata_path}')

# Verificar tamaño del archivo
size_mb = model_path.stat().st_size / 1024 / 1024
print(f'\nTamaño del modelo: {size_mb:.2f} MB')
print(f'Features: {len(feature_names)}')

Modelo guardado: ../models/lgbm_credit_risk.pkl
Metadatos guardados: ../models/model_metadata.json

Tamaño del modelo: 1.17 MB
Features: 72


## 3. Verificar que el modelo cargado produce los mismos resultados

In [4]:
# Cargar el modelo desde disco y verificar que produce exactamente los mismos resultados
model_loaded = joblib.load(model_path)

y_proba_loaded = model_loaded.predict_proba(X_test)[:, 1]
roc_auc_loaded = roc_auc_score(y_test, y_proba_loaded)

print(f'ROC-AUC modelo original: {roc_auc:.6f}')
print(f'ROC-AUC modelo cargado:  {roc_auc_loaded:.6f}')
print(f'Resultados idénticos: {np.allclose(y_proba, y_proba_loaded)}')
print()
print('✓ El modelo guardado en disco produce exactamente los mismos resultados.')

ROC-AUC modelo original: 0.756110
ROC-AUC modelo cargado:  0.756110
Resultados idénticos: True

✓ El modelo guardado en disco produce exactamente los mismos resultados.
